## 基本环境 · Basic setup

首次打开运行下面 3 个 cell。它们做的事:
1. 把工作目录切到所在的代码树根 (`solutions/` 或 `tutorials/`)，
   这样 `from attention.mha import ...` 这种导入能直接生效。
2. 启用 `autoreload`，编辑 .py 文件保存后 notebook 里立刻可用，不用重启 kernel。
3. 设 `LAYERNORM_TYPE=torch`，避免 CUDA-only 算子的 import 失败。

First time you open the notebook, run the 3 cells below: cd to the tree root (whichever of `solutions/` or `tutorials/` this notebook lives in), turn on autoreload, force the pure-PyTorch LayerNorm path.

In [ ]:
import os, sys

# Walk up from the notebook's CWD until we find a directory named
# `solutions` or `tutorials`. Works no matter which tree the student
# opened. 不论 notebook 位于 solutions/ 还是 tutorials/ 都能正确定位。
ROOTS = {'solutions', 'tutorials'}
if os.path.basename(os.getcwd()) not in ROOTS:
    while os.path.basename(os.getcwd()) not in ROOTS and os.getcwd() != '/':
        os.chdir('..')
    if os.path.basename(os.getcwd()) not in ROOTS:
        # Fallback: maybe we were started at the repo root.
        if os.path.isdir('tutorials'):
            os.chdir('tutorials')
        elif os.path.isdir('solutions'):
            os.chdir('solutions')

assert os.path.basename(os.getcwd()) in ROOTS, (
    f'could not locate solutions/ or tutorials/ from {os.getcwd()}')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')
print('cwd =', os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
# Folder where the pairformer chapter's reference .pt files live
control_folder = 'pairformer/control_values'
assert os.path.isdir(control_folder), f'missing {control_folder}'

# 第 2 章 · Pairformer

## 为什么需要 Pairformer

结构预测最关键的问题: **判断哪两个残基会在 3D 中接近**。如果模型只对单序列做Transformer，永远拿不到 pair (i, j) 的几何先验。AF2 用 Evoformer 同时跑MSA + pair 两个表示，反复交换信息；**AF3 简化为 Pairformer**: pair 表示是主角，MSA 表示只在 MSAModule 里短暂出现，把信息汇入 pair 后就消失。

Pairformer 让 $z_{ij}$ 满足结构上的[三角不等式约束](https://en.wikipedia.org/wiki/Triangle_inequality):

$$\forall i, j, k: \quad |z_{ij}| \le |z_{ik}| + |z_{kj}|$$

实现这点的关键是**让 (i, j) 的更新明确依赖第三个 token k** —— 这就是「三角」一词的来源。具体有 4 种三角算子，本章逐个手写。

## 本章模块

| 文件 | 类 | 算法 | 简述 |
|---|---|---|---|
| `triangle_ops.py` | `OuterProductMean` | Alg 10 | MSA → pair |
| `triangle.py` | `TriangleMultiplication{Outgoing,Incoming}` | Alg 11 / 12 | pair 上的外积式更新 |
| `triangle.py` | `TriangleAttention` | Alg 13 / 14 | pair 上沿一条 residue 轴的注意力 |
| `msa_stack.py` | `MSAPairWeightedAveraging` | (MSAModule 内) | pair → MSA 反向通道 |
| `pair_stack.py` | `PairformerBlock` | Alg 17 | 把以上 4 个 + pair_transition + 可选 single update 串起来 |

## 前置

本章会反复用到第 1 章实现的 `Linear` / `LinearNoBias` / `LayerNorm` / `Transition` /`AttentionPairBias`。如果第 1 章测试还没全绿，回去补完。

## 2.1 OuterProductMean (Algorithm 10) — MSA → pair

AF3 的 MSAModule 反复要做的事: **从 MSA 中提炼共进化信号塞回 pair 通道**。共进化(coevolution) 是结构预测最古老的信号 —— 如果残基 i 和 j 在 MSA 的所有同源序列里总是「一起变」，它们在 3D 中很可能接触。

OuterProductMean 是这件事的最简单实现:

$$z_{ij} \;+\!\!= \;W_o \, \mathrm{flat}\Big(\frac{1}{N_{\text{eff}}(i,j)} \sum_{s=1}^{N_{\text{msa}}} \mathrm{mask}_{si}\mathrm{mask}_{sj}\;a_{si} \otimes b_{sj}\Big)$$

其中 $a, b$ 是 MSA 张量经 LayerNorm + 两路线性 (`linear_1` / `linear_2`) 投到`c_hidden` 后的产物。**外积**把两个 (c_hidden,) 向量变成 (c_hidden × c_hidden) 矩阵 ——捕获每对位置 (i, j) 在通道维上的所有交互组合。然后沿 MSA 维 (s) 加权平均，flatten再投回 `c_z`。

### 实现要点

- **einsum**: 高效写法 `'...bac, ...dae -> ...bdce'` 沿 MSA 维 a 求和。
- **mask 归一化**: $N_\text{eff}(i, j) = \sum_s \mathrm{mask}_{si} \mathrm{mask}_{sj} + \epsilon$，  保证至少一个有效序列时不除零。
- **输出投影 `linear_out` 用 `init="final"`** —— 即零初始化，pair 更新起手是恒等。

**任务**: 打开 `pairformer/triangle_ops.py` 填三处 TODO ——`OuterProductMean.__init__` (4 个子模块) / `_opm` (einsum + flatten + 投影) / `_forward` (整个流程)。

In [ ]:
from pairformer.triangle_ops import OuterProductMean
from pairformer.control_values.pairformer_checks import (
    c_m, c_z, c_hidden, no_heads_pair, test_inputs,
    test_module_shape, test_module_method, test_module_forward,
)

opm = OuterProductMean(c_m=c_m, c_z=c_z, c_hidden=c_hidden)
test_module_shape(opm, 'outer_product_mean', control_folder)
test_module_method(
    opm, 'outer_product_mean',
    inputs=(test_inputs['m'], test_inputs['msa_mask']),
    output_names='out',
    control_folder=control_folder,
    method=lambda m, mask: opm(m, mask=mask),
)
print('OuterProductMean ✓')

## 2.2 TriangleMultiplication (Algorithms 11 & 12)

Pair 表示 $z_{ij}$ 的「三角」更新里，最便宜的一种: 不跑注意力，直接做**带门控的外积**。

### 几何直观

想象 3 个 token i, j, k 之间存在一个三角形。$z_{ik}$ 与 $z_{jk}$ 都包含关于 k 的信息 —— 那么 $z_{ij}$ 也应该能通过"绕一圈"得到约束:

- **Outgoing** (Alg 11): $z_{ij} \,+\!\!=\, \sum_k a_{ik} \odot b_{jk}$  
  从 i 出发、k 是「中介」，每条三角形边 (i, k) 与 (j, k) 提供一对乘子。
- **Incoming** (Alg 12): $z_{ij} \,+\!\!=\, \sum_k a_{ki} \odot b_{kj}$  
  反过来，从 k 流入 i 和 j。

Pairformer 一个 block 同时跑这两条，让 $z_{ij}$ 在 i / j 两个角色上都收到信号。

### 计算流程

1. LN(z)  
2. 每路两个线性 + sigmoid 门: $a = \sigma(W_{ag} z) \odot W_{ap} z$，同样得 $b$
3. mask 一下 (`mask * a`, `mask * b`)
4. **核心 trick** —— 把通道维提到最前 (`permute_final_dims`)，再 `torch.matmul`，   让 BLAS 帮你算 $\sum_k$。Outgoing / Incoming 的差别在于 a / b 的 permute 方式不同。
5. LN(combined) + 投回 c_z + sigmoid 门 (`linear_g`)

**任务**: 在 `pairformer/triangle.py` 填:

- `BaseTriangleMultiplicativeUpdate.__init__` — 7 个共享子模块 (4 个 LN/Linear + 2 LN + sigmoid)
- `TriangleMultiplicativeUpdate.__init__` — 追加 4 个 a/b projection
- `_combine_projections` — permute + matmul + permute 回来 (附带可选 inplace_chunk)
- `forward` — 串起 LN → 两路门控投影 → combine → out projection → 输出门

子类 `TriangleMultiplicationOutgoing` / `Incoming` 不需要写 —— 它们用 `partialmethod`固定 `_outgoing=True/False`。

In [ ]:
from pairformer.triangle import (
    TriangleMultiplicationOutgoing, TriangleMultiplicationIncoming,
)

for variant, cls in [
    ('triangle_mul_out', TriangleMultiplicationOutgoing),
    ('triangle_mul_in',  TriangleMultiplicationIncoming),
]:
    mod = cls(c_z=c_z, c_hidden=c_hidden)
    test_module_shape(mod, variant, control_folder)
    test_module_method(
        mod, variant,
        inputs=(test_inputs['z'], test_inputs['pair_mask']),
        output_names='out',
        control_folder=control_folder,
        method=lambda z, pm, mod=mod: mod(z, mask=pm),
    )
print('TriangleMultiplication (outgoing + incoming) ✓')

## 2.3 TriangleAttention (Algorithms 13 & 14)

刚才的 TriangleMultiplication 只能学到"哪两条三角形边相乘起来强"。**TriangleAttention**更进一步: 让模型**主动选择**通过哪个第三个节点 k 来连接 (i, j)。

把 pair 表示 $z_{ij}$ 想象成一张 NxN 的图。沿一行 (固定 i, 遍历 j) 跑多头注意力，**注意力分数本身又来自另一行的 pair 张量** —— 这就是「triangle」的味道。

### 两种变种

- **Around starting node** (Alg 13, `starting=True`): 在 z[i, :, :] 这一**行**里跑 attention，  每行内部 i 固定、j 是 query / key/value 序列。
- **Around ending node** (Alg 14, `starting=False`): 在 z[:, j, :] 这一**列**里跑 attention。

代码实现共用同一个类，**ending 版用 `transpose(-2, -3)` 把列翻成行就行**。

### 实现细节

- 通道维 LayerNorm。
- 一个 `linear: c_in → no_heads`，把 pair 投成每头一个 bias 标量，再 `permute_final_dims`  把头维提前 + `unsqueeze(-4)` 给一个假 row 维，让它能广播加到 attention 分数上。
- mask 走 `mask_bias = inf * (mask - 1)`，softmax 自然把无效位置压到 0。
- `self.mha` 是 OpenFold 风格 `Attention` (与第 1 章 AF3 那个不一样!)，接受 `biases=[...]` 列表。

**任务**: 填 `TriangleAttention.__init__` + `forward`。

In [ ]:
from pairformer.triangle import TriangleAttention

tri_att = TriangleAttention(
    c_in=c_z, c_hidden=c_hidden, no_heads=no_heads_pair, starting=True,
)
test_module_shape(tri_att, 'triangle_attention_start', control_folder)
test_module_method(
    tri_att, 'triangle_attention_start',
    inputs=(test_inputs['z'], test_inputs['pair_mask']),
    output_names='out',
    control_folder=control_folder,
    method=lambda z, pm: tri_att(z, mask=pm),
)
print('TriangleAttention ✓')

## 2.4 MSAPairWeightedAveraging

OuterProductMean 是 **MSA → pair** 通道；它的兄弟 **pair → MSA** 是这里的`MSAPairWeightedAveraging`。它出现在 MSAModule 内部，每个 block 跑一次:

$$m'_{si} = \sum_h g_{si}^h \;\sum_j \mathrm{softmax}_j(b_{ij}^h) \cdot v_{sj}^h$$

解读:

- $b_{ij}^h$ 来自 pair 张量经 LN + linear 投到每头一个标量；它告诉每个 MSA 序列  「在更新位置 i 时，应该多看位置 j」。
- $v_{sj}^h$ 是 MSA 在位置 j 经线性投影得到的 value (跨 MSA 序列 s 共享 b 权重)。
- 沿 j 加权平均，再用 sigmoid 门 $g_{si}^h$ 收尾 (zero-init，起手关闭)。

**为什么重要**: 单跑 OuterProductMean MSA 信息只能流向 pair，再不回头。加上 MSAPairWeightedAveraging，pair 学到的关系能反过来精修 MSA，让 OPM 下一轮的输入更好。

**任务**: 在 `pairformer/msa_stack.py` 填 `MSAPairWeightedAveraging.__init__` 和 `.forward`。

In [ ]:
from pairformer.msa_stack import MSAPairWeightedAveraging

mpwa = MSAPairWeightedAveraging(c_m=c_m, c=c_hidden, c_z=c_z, n_heads=no_heads_pair)
test_module_shape(mpwa, 'msa_pair_weighted_avg', control_folder)
test_module_forward(
    mpwa, 'msa_pair_weighted_avg',
    inputs=(test_inputs['m'], test_inputs['z']),
    output_names='out',
    control_folder=control_folder,
)
print('MSAPairWeightedAveraging ✓')

## 2.5 PairformerBlock (Algorithm 17 — 一整块)

这一节把前面 4 个三角操作 + Transition + 单序列更新拼成 AF3 主干的最小重复单元。Algorithm 17 一个 block 的伪代码:

```text
  z += TriangleMultiplicationOutgoing(z)
  z += TriangleMultiplicationIncoming(z)
  z += TriangleAttention starting_node(z)
  z += TriangleAttention ending_node(z)         ← 通过物理转置实现
  z += pair_transition(z)
  if c_s > 0:                                    ← 单序列分支
      s += AttentionPairBias(a=s, s=None, z=z)
      s += single_transition(s)
```

### 几个工程细节

- 前两个三角乘走 `inplace_safe=True, _add_with_inplace=True` 路径，  把 `z += op(z)` 做成融合操作 (省一份 z 的 buffer)。
- 后两个 attention 用普通 `z = z + op(z)`。ending_node 通过先转置再调  `tri_att_end(z.T)` 再转回来实现 Algorithm 14 —— 模型只学一份 starting 权重。
- 单序列分支用第 1 章的 `AttentionPairBias` 但 `has_s=False` (LN 而非 AdaLN)，  并把 LN over z 的 offset 打开 (`create_offset_ln_z=True`) —— Pairformer 习惯。

### 测试简化

本格用 `c_s=0` 跳过单序列分支，只验证 pair 通道的更新链路是对的。完整 `c_s > 0` 的 pairformer 会在端到端推理时由 Protenix 装配出来。

**任务**: 填 `PairformerBlock.__init__` 和 `.forward` 两处 TODO。

In [ ]:
from pairformer.pair_stack import PairformerBlock

block = PairformerBlock(
    n_heads=no_heads_pair,
    c_z=c_z, c_s=0,
    c_hidden_mul=c_hidden,
    c_hidden_pair_att=c_hidden,
    no_heads_pair=no_heads_pair,
    num_intermediate_factor=2,
    dropout=0.0,
)
for sub in (block.tri_att_start.mha, block.tri_att_end.mha):
    if hasattr(sub, 'use_efficient_implementation'):
        sub.use_efficient_implementation = False

test_module_shape(block, 'pairformer_block_no_single', control_folder)
test_module_method(
    block, 'pairformer_block_no_single',
    inputs=(None, test_inputs['z'], test_inputs['pair_mask']),
    output_names='z_out',
    control_folder=control_folder,
    method=lambda s, z, pm: block(s, z, pair_mask=pm)[1],
)
print('PairformerBlock ✓')

## 章节小结

完成本章后你掌握了 AF3 主干的 5 个核心三角算子:

| 方向 | 算子 | 文件 |
|---|---|---|
| MSA → pair | OuterProductMean | `pairformer/triangle_ops.py` |
| pair → MSA | MSAPairWeightedAveraging | `pairformer/msa_stack.py` |
| pair → pair (乘法) | TriangleMul Out + In | `pairformer/triangle.py` |
| pair → pair (注意力) | TriangleAttention Start + End | `pairformer/triangle.py` |
| 集成 | PairformerBlock | `pairformer/pair_stack.py` |

把 ~48 个 PairformerBlock 堆叠起来就是 AF3 主干 (我们的 tiny 配置只用 8 个)。每个 block都会让 pair 张量更接近"满足三角形不等式的实际几何距离矩阵"。**下一站**: Feature embedding 把原子 / 残基特征翻译成 trunk 能吃的张量；之后 Diffusion 用 trunk 学到的几何信号反向去噪出真坐标。